<div style="border: 3px solid #0078B8; padding: 15px; border-radius: 12px; background-color: #00B6DA; text-align: center;">
  <h1 style="color: #000000;">Projet - Analyse du marché immobilier en France</h1>
</div>

### Identifier les zones attractives pour l'investissement locatif et patrimonial
Objectif du projet : analyser le marché immobilier français afin d'identifier les régions et communes les plus intéressantes pour investir dans le cadre : 
- d'un investissement locatif → recherche de rendement
- d'un investissement patrimonial → valorisation long terme

Question : Où acheter en france aujourd'hui pour optimiser rentabilité et/ou potentiel de valorisation ? 

Contexte : L'investissement immobilier est l'un des placements préférés des Français. Cependant, le marché est hétérogène : 
- forte disparité des prix entre les régions, 
- tension locatives variables, 
- dynamiques démographiques différentes, 
- rendements très contrastés selon les communes.

Une analyse basée sur la donnée permet de dépasser l'intuition et d'identifier des opportunités plus objectivement

#### Données utilisées 
Ce projet s'appuis sur : 
- Valeurs foncières → https://www.data.gouv.fr/datasets/demandes-de-valeurs-foncieres 
- Loyer 2022 → https://www.data.gouv.fr/datasets/carte-des-loyers-indicateurs-de-loyers-dannonce-par-commune-en-2022 
- Loyer 2023 → https://www.data.gouv.fr/datasets/carte-des-loyers-indicateurs-de-loyers-dannonce-par-commune-en-2023
- Loyer 2024 → https://www.data.gouv.fr/datasets/carte-des-loyers-indicateurs-de-loyers-dannonce-par-commune-en-2024 
- Population 2020 → https://www.insee.fr/fr/statistiques/6683035?sommaire=6683037 
- Population 2021 → https://www.insee.fr/fr/statistiques/7739582?sommaire=7728826 
- Population 2022 → https://www.insee.fr/fr/statistiques/8290591?sommaire=8290669
- Population 2023 → https://www.insee.fr/fr/statistiques/8680726?sommaire=8681011

Ce notebook a pour objectif de construire un dataset analytique consolidé à partir de sources multiples (DVF, loyers DHUP, INSEE).
Le dataset final est exporté et utilisé dans un second notebook dédié à l’analyse.

In [1173]:
#Importation des librairies
import pandas as pd
import glob
import re
import numpy as np
import plotly.express as px 
import seaborn as sns
import matplotlib.pyplot as plt

<div style="background-color: #0078B8; border-radius: 12px;" >
<h2 style="margin: auto; padding: 20px; color:#000000; ">1 - Analyse exploratoire et nettoyage des données</h2>
</div>

<div style="background-color: #00B6DA; border-radius: 12px;" >
<h3 style="margin: auto; padding: 20px; color:#000000; ">1.1 - Valeurs Foncières </h3>
</div>

In [1174]:
#Lire uniquement l'en-tête (0 ligne de data)
df_columns = pd.read_csv(
    "ValeursFoncieres-2024.txt",
    sep="|",
    encoding="utf-8",
    nrows=0)

#Afficher les colonnes
print("Nombre de colonnes :", len(df_columns.columns))
print("\nListe des colonnes :\n")
for col in df_columns.columns:
    print(col)

Nombre de colonnes : 43

Liste des colonnes :

Identifiant de document
Reference document
1 Articles CGI
2 Articles CGI
3 Articles CGI
4 Articles CGI
5 Articles CGI
No disposition
Date mutation
Nature mutation
Valeur fonciere
No voie
B/T/Q
Type de voie
Code voie
Voie
Code postal
Commune
Code departement
Code commune
Prefixe de section
Section
No plan
No Volume
1er lot
Surface Carrez du 1er lot
2eme lot
Surface Carrez du 2eme lot
3eme lot
Surface Carrez du 3eme lot
4eme lot
Surface Carrez du 4eme lot
5eme lot
Surface Carrez du 5eme lot
Nombre de lots
Code type local
Type local
Identifiant local
Surface reelle bati
Nombre pieces principales
Nature culture
Nature culture speciale
Surface terrain


Liste des colonnes à garder : 
- Localisation (clé pour jointure) → Code departement; Code commune; Commune; Code postal
- Caractéristique du bien (segmentation) → Type local; Code type local; Nombre de pieces principales
- Prix et surfaces (pour prix/m²) → Valeur fonciere; Surface reelle bati, Surface terrain
- Date et filtre qualité → Date mutation; Nature mutation (pour garder seulement"Vente")

In [1175]:
liste_cols = [
    "Date mutation",
    "Nature mutation",
    "Valeur fonciere",
    "Code departement",
    "Code commune",
    "Commune",
    "Code postal",
    "Type local",
    "Code type local",
    "Surface reelle bati",
    "Surface terrain",
    "Nombre pieces principales"]

In [1176]:
#Conversion des fichiers TXT en gardant uniquement les colonnes d’intérêt + fusion

#Liste des fichiers à traiter
fichiers = sorted(glob.glob("ValeursFoncieres-20*.txt"))
#Liste pour stocker les DataFrames de chaque année
dfs_VF = []
#Boucle de traitement de chaque fichier
for fichier in fichiers:
    annee = int(fichier.split("-")[1].split(".")[0])
    print(f"Lecture {fichier} (année {annee})")
    #Lecture du fichier en ne gardant que les colonnes d'intérêt
    df_temp = pd.read_csv(
        fichier,
        sep="|",
        encoding="utf-8",
        usecols=liste_cols,
        low_memory=False)
    # Ajout de la colonne année_source
    df_temp["annee_source"] = annee
    # Stockage dans la liste
    dfs_VF.append(df_temp)
# Fusion finale
df_rawVF = pd.concat(dfs_VF, ignore_index=True)
print("Fusion terminée :", df_rawVF.shape)

Lecture ValeursFoncieres-2021.txt (année 2021)
Lecture ValeursFoncieres-2022.txt (année 2022)
Lecture ValeursFoncieres-2023.txt (année 2023)
Lecture ValeursFoncieres-2024.txt (année 2024)
Fusion terminée : (16650659, 13)


In [1177]:
df_rawVF.head()

,Date mutation,Nature mutation,Valeur fonciere,Code postal,Commune,Code departement,Code commune,Code type local,Type local,Surface reelle bati,Nombre pieces principales,Surface terrain,annee_source
0,05/01/2021,Vente,"185000,00",1370.0,VAL-REVERMONT,01,426,3.0,Dépendance,0.0,0.0,2410.0,2021
1,05/01/2021,Vente,"185000,00",1370.0,VAL-REVERMONT,01,426,1.0,Maison,97.0,5.0,2410.0,2021
2,06/01/2021,Vente,"10,00",1290.0,BEY,01,42,NaN,NaN,NaN,NaN,530.0,2021
3,04/01/2021,Vente,"204332,00",1310.0,BUELLAS,01,65,1.0,Maison,88.0,4.0,866.0,2021
4,06/01/2021,Vente,"320000,00",1250.0,MONTAGNAT,01,254,3.0,Dépendance,0.0,0.0,1426.0,2021


In [1178]:
df_rawVF.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 16650659 entries, 0 to 16650658
Data columns (total 13 columns):
 #   Column                     Dtype  
---  ------                     -----  
 0   Date mutation              object 
 1   Nature mutation            object 
 2   Valeur fonciere            object 
 3   Code postal                float64
 4   Commune                    object 
 5   Code departement           object 
 6   Code commune               int64  
 7   Code type local            float64
 8   Type local                 object 
 9   Surface reelle bati        float64
 10  Nombre pieces principales  float64
 11  Surface terrain            float64
 12  annee_source               int64  
dtypes: float64(5), int64(2), object(6)
memory usage: 1.6+ GB


In [1179]:
#Afficher les valeurs distinctes des colonnes 'Nature mutation' et 'Type local'
print("Nature mutation :", df_rawVF['Nature mutation'].unique())
print("Type local :", df_rawVF['Type local'].unique())

Nature mutation : ['Vente' "Vente en l'état futur d'achèvement" 'Echange'
 'Vente terrain à bâtir' 'Adjudication' 'Expropriation']
Type local : ['Dépendance' 'Maison' nan 'Appartement'
 'Local industriel. commercial ou assimilé']


In [1180]:
#Création copie df_rawVF avec filtre des colonnes 'Nature mutation' (Vente) et 'Type local' (Maison et Appartement)
df_cleanVF = df_rawVF[
    (df_rawVF['Nature mutation'] == 'Vente') &
    (df_rawVF['Type local'].isin(['Maison', 'Appartement']))
].copy()

print("Après filtre ventes + résidentiel :", df_cleanVF.shape)
df_cleanVF["Type local"].value_counts(dropna=False)

Après filtre ventes + résidentiel : (4953899, 13)


Type local
Maison         2716561
Appartement    2237338
Name: count, dtype: int64

In [1181]:
#Convertion des types de données pour les colonnes 'Date mutation' et 'Valeur fonciere'

#'Date mutation' object → datetime (format français JJ/MM/AAAA)
df_cleanVF["Date mutation"] = pd.to_datetime(
    df_cleanVF["Date mutation"],
    dayfirst=True,
    errors="coerce")

print("Dates invalides :", df_cleanVF["Date mutation"].isna().sum())

#'Valeur fonciere' object → float
df_cleanVF["Valeur fonciere"] = (
    df_cleanVF["Valeur fonciere"]
    .astype(str)
    .str.replace(",", ".", regex=False))

df_cleanVF["Valeur fonciere"] = pd.to_numeric(df_cleanVF["Valeur fonciere"], errors="coerce")
print("Valeur fonciere invalides :", df_cleanVF["Valeur fonciere"].isna().sum())

Dates invalides : 0
Valeur fonciere invalides : 22346


In [1182]:
#Suppression des lignes non exploitables (valeur fonciere et date mutation indoivent être > 0)
df_cleanVF = df_cleanVF[
    (df_cleanVF["Valeur fonciere"].notna()) & (df_cleanVF["Valeur fonciere"] > 0) &
    (df_cleanVF["Surface reelle bati"].notna()) & (df_cleanVF["Surface reelle bati"] > 0)
].copy()

In [1183]:
#Vérification des valeurs manquantes
print("Nombre de valeurs manquantes :", df_cleanVF.isna().sum())

Nombre de valeurs manquantes : Date mutation                      0
Nature mutation                    0
Valeur fonciere                    0
Code postal                      200
Commune                            0
Code departement                   0
Code commune                       0
Code type local                    0
Type local                         0
Surface reelle bati                0
Nombre pieces principales          0
Surface terrain              1826835
annee_source                       0
dtype: int64


In [1184]:
#Calcul du prix au mètre carré (Valeur fonciere / Surface reelle bati) 
df_cleanVF["prix_m2"] = df_cleanVF["Valeur fonciere"] / df_cleanVF["Surface reelle bati"]
df_cleanVF["prix_m2"].describe(percentiles=[0.01, 0.05, 0.5, 0.95, 0.99])

count    4.931104e+06
mean     3.953129e+04
std      4.199410e+05
min      1.010101e-03
1%       1.927711e+02
5%       7.000000e+02
50%      2.692500e+03
95%      2.000000e+04
99%      6.048383e+05
max      4.663156e+07
Name: prix_m2, dtype: float64

- 4 931 104 transactions → bon volume
- prix m² minimum 0.001 € → soit un bien a été vendu pour 1 €; soit une erreur de saisie
- prix m² maximum 46 631 560 € → impossible surement une erreur de saisie, surface ou valeur foncière
- moyenne prix m² 39 531 € → la moyenne est tirée vers le haut par les outliers extrêmes 
- médiane prix m² 2 692 € → vrai prix central du marché français

L'analyse montre une forte asymétrie des prix du m² en France, avec des valeurs extrêmes très élevées et très faibles. Afin de limiter l'influence des erreurs ou transactions atypiques, je vais appliquer un filtrage avec des prix du m² entre 300 € et 20 000 €, bornes cohérentes avec les percentiles observés.

In [1185]:
#Filtre des valeurs aberrantes (prix au mètre carré < 300 € et > 20 000 €)
df_cleanVF = df_cleanVF[(df_cleanVF["prix_m2"] >= 300) & (df_cleanVF["prix_m2"] <= 20000)].copy()

In [1186]:
#Statistiques descriptives de la surface réelle bâtie
df_cleanVF["Surface reelle bati"].describe(percentiles=[0.01, 0.05, 0.5, 0.95, 0.99])

count    4.611619e+06
mean     8.353828e+01
std      4.667402e+01
min      1.000000e+00
1%       1.700000e+01
5%       2.500000e+01
50%      7.700000e+01
95%      1.650000e+02
99%      2.400000e+02
max      3.160000e+03
Name: Surface reelle bati, dtype: float64

- médiane surface 77 m² → cohérent pour la France, mélange maisons et appartements
- moyenne sruface 83 m² → proche de la médiane 
- 95% font moins de 165 m² → très cohérent
- 1% font plus de 240 m² → pas forcement aberrant, probablement des grandes maisons
- surface maximum 3 160 m² → valeur atypique, peut être un hôtel, immeuble, erreur de saisie
- surface minimum 1 m² → inhabitable, probablement une erreur de saisie

L'analyse montre que les surface semble cohérente avec le marché français mais des biens atypiques ou erreurs de saisie existe. Afin d'exclure ces valeurs aberrantes je vais appliquer un filtrage avec des surface entre 10 m² et 500 m².

In [1187]:
#Filtre des valeurs aberrantes (surface réelle bâtie < 10 m² et > 500 m²)
df_cleanVF = df_cleanVF[(df_cleanVF["Surface reelle bati"] >= 10) & (df_cleanVF["Surface reelle bati"] <= 500)].copy()

In [1188]:
#Vérification des valeurs nulles dans "Nombre pieces principales"
print("Valeurs nulles :", df_cleanVF["Nombre pieces principales"].isin([0]).sum())

Valeurs nulles : 6792


In [1189]:
#Affichage des lignes avec "Nombre pieces principales" < 0
df_cleanVF[df_cleanVF["Nombre pieces principales"] < 0].head()

,Date mutation,Nature mutation,Valeur fonciere,Code postal,Commune,Code departement,Code commune,Code type local,Type local,Surface reelle bati,Nombre pieces principales,Surface terrain,annee_source,prix_m2


In [1190]:
#Statistiques descriptives du nombre de pièces principales
df_cleanVF["Nombre pieces principales"].describe(percentiles=[0.01, 0.05, 0.5, 0.95, 0.99])

count    4.606632e+06
mean     3.517142e+00
std      1.604064e+00
min      0.000000e+00
1%       1.000000e+00
5%       1.000000e+00
50%      3.000000e+00
95%      6.000000e+00
99%      8.000000e+00
max      1.980000e+02
Name: Nombre pieces principales, dtype: float64

- 6 792 valeurs nulles dans nombre de pieces principales → négligeable sur les 4.6 millions de lignes (on peut supprimer)
- médiane 3 pièces → cohérent pour la France 
- 99% des biens ont 8 pièces ou moins → les valeurs ne semblent pas aberrantes 
- maximum de pièces 198 → valeur aberrantes, peut être un hôtel, immeuble, erreur de saisie

Donc après analyse je vais supprimer les valeurs nulles ainsi que les valeurs supérieurs à 20 pièces.

In [1191]:
df_cleanVF = df_cleanVF[df_cleanVF["Nombre pieces principales"] > 0]
df_cleanVF = df_cleanVF[df_cleanVF["Nombre pieces principales"] <= 20]

In [1192]:
df_cleanVF.info()

<class 'pandas.core.frame.DataFrame'>
Index: 4599576 entries, 1 to 16650656
Data columns (total 14 columns):
 #   Column                     Dtype         
---  ------                     -----         
 0   Date mutation              datetime64[ns]
 1   Nature mutation            object        
 2   Valeur fonciere            float64       
 3   Code postal                float64       
 4   Commune                    object        
 5   Code departement           object        
 6   Code commune               int64         
 7   Code type local            float64       
 8   Type local                 object        
 9   Surface reelle bati        float64       
 10  Nombre pieces principales  float64       
 11  Surface terrain            float64       
 12  annee_source               int64         
 13  prix_m2                    float64       
dtypes: datetime64[ns](1), float64(7), int64(2), object(4)
memory usage: 526.4+ MB


In [1193]:
#Création clé INSEE complète
df_cleanVF["Code departement"] = df_cleanVF["Code departement"].astype(str).str.strip()
df_cleanVF["Code commune"] = df_cleanVF["Code commune"].astype(str).str.strip()

# Code commune sur 3 chiffres
df_cleanVF["Code commune_3"] = df_cleanVF["Code commune"].str.zfill(3)

# Département : cas général (2 chiffres) + cas Corse (2A/2B)
df_cleanVF["Code_dep_norm"] = df_cleanVF["Code departement"].where(
    df_cleanVF["Code departement"].isin(["2A", "2B"]),
    df_cleanVF["Code departement"].str.zfill(2))

df_cleanVF["Code_INSEE"] = df_cleanVF["Code_dep_norm"] + df_cleanVF["Code commune_3"]

<div style="background-color: #00B6DA; border-radius: 12px;" >
<h3 style="margin: auto; padding: 20px; color:#000000; ">1.2 - Loyers </h3>
</div>

In [1194]:
def load_clean_loyer(path, segment):
    df = pd.read_csv(path, sep=";", encoding="latin-1")

    #Colonnes numériques à nettoyer
    num_cols = ["loypredm2", "lwr.IPm2", "upr.IPm2", "R2_adj"]
    for c in num_cols:
        df[c] = (
            df[c].astype(str)
            .str.replace("\u202f", "", regex=False)
            .str.replace("\u00a0", "", regex=False)
            .str.replace("\t", "", regex=False)
            .str.replace(" ", "", regex=False)
            .str.replace(",", ".", regex=False))
        df[c] = pd.to_numeric(df[c], errors="coerce")

    #Code INSEE propre
    df["INSEE_C"] = df["INSEE_C"].astype(str).str.zfill(5)

    #Création colonne segment
    df["segment_loyer"] = segment

    #Année extraite du nom de fichier (ex: ...-2023.csv)
    m = re.search(r"(20\d{2})", path)
    df["annee"] = int(m.group(1)) if m else None

    return df

In [1195]:
#Mapping des préfixes de fichiers vers les segments correspondants
mapping = {
    "pred-app-mef-dhup": "APP",
    "pred-app12-mef-dhup": "APP12",
    "pred-app3-mef-dhup": "APP3",
    "pred-mai-mef-dhup": "MAI",}

dfs = []

for base, segment in mapping.items():
    #Récupère toutes les années disponibles pour ce segment
    paths = sorted(glob.glob(f"{base}-20*.csv"))
    for path in paths:
        print("Lecture :", path)
        dfs.append(load_clean_loyer(path, segment))

df_cleanLoyer = pd.concat(dfs, ignore_index=True)

print("Shape final loyers :", df_cleanLoyer.shape)
print(df_cleanLoyer.groupby(["annee", "segment_loyer"]).size())

Lecture : pred-app-mef-dhup-2022.csv
Lecture : pred-app-mef-dhup-2023.csv
Lecture : pred-app-mef-dhup-2024.csv
Lecture : pred-app12-mef-dhup-2022.csv
Lecture : pred-app12-mef-dhup-2023.csv
Lecture : pred-app12-mef-dhup-2024.csv
Lecture : pred-app3-mef-dhup-2022.csv
Lecture : pred-app3-mef-dhup-2023.csv
Lecture : pred-app3-mef-dhup-2024.csv
Lecture : pred-mai-mef-dhup-2022.csv
Lecture : pred-mai-mef-dhup-2023.csv
Lecture : pred-mai-mef-dhup-2024.csv
Shape final loyers : (419640, 15)
annee  segment_loyer
2022   APP              34980
       APP12            34980
       APP3             34980
       MAI              34980
2023   APP              34970
       APP12            34970
       APP3             34970
       MAI              34970
2024   APP              34960
       APP12            34960
       APP3             34960
       MAI              34960
dtype: int64


In [1196]:
# Vérifier années présentes
print("Années :", sorted(df_cleanLoyer["annee"].dropna().unique()))

# Vérifier segments présents par année
df_cleanLoyer.pivot_table(index="annee", columns="segment_loyer", values="INSEE_C", aggfunc="count")

Années : [np.int64(2022), np.int64(2023), np.int64(2024)]


segment_loyer,APP,APP12,APP3,MAI
annee,,,,
2022,34980,34980,34980,34980
2023,34970,34970,34970,34970
2024,34960,34960,34960,34960


In [1197]:
#Vérification des valeurs nulles dans "loypredm2"
print("Valeurs nulles :", df_cleanLoyer["loypredm2"].isna().sum())

Valeurs nulles : 0


In [1198]:
#Statistiques descriptives de "loypredm2" par segment
df_cleanLoyer.groupby("segment_loyer")["loypredm2"].describe()

,count,mean,std,min,25%,50%,75%,max
segment_loyer,,,,,,,,
APP,104910.0,9.709651,2.300673,5.467790,8.164651,9.264396,10.713990,37.186719
APP12,104910.0,11.605639,2.440392,7.338075,9.971867,11.205208,12.661848,39.415339
APP3,104910.0,8.385022,2.167748,4.967488,6.981366,7.938061,9.199309,37.166954
MAI,104910.0,8.551159,2.256696,4.902666,7.117082,8.055042,9.359043,30.967737


In [1199]:
#Statistiques descriptives globales de "loypredm2"
df_cleanLoyer["loypredm2"].describe(percentiles=[0.01,0.05,0.5,0.95,0.99])

count    419640.000000
mean          9.562868
std           2.628955
min           4.902666
1%            5.731279
5%            6.348515
50%           9.086907
95%          14.279298
99%          18.249792
max          39.415339
Name: loypredm2, dtype: float64

- APP médiane 8.97 €/m²; max 33 €/m² → cohérent
- APP12 médiane > APP3 → les plus petites surfaces ont un loyer au m² plus élevé en général
- MAI médiane 7.73 €/m² → légèrement inférieur aux appartements ce qui est cohérent
- Distribution globale → pas de valeur aberrante 


In [1200]:
#Afficher les valeurs distinctes des colonnes 'TYPPRED' et 'Type local'
print("TYPPRED :", df_cleanLoyer['TYPPRED'].unique())

TYPPRED : ['maille' 'commune' 'EPCI']


In [1201]:
#Afficher la répartition des valeurs de 'TYPPRED'
df_cleanLoyer["TYPPRED"].value_counts(normalize=True)

TYPPRED
maille     0.889417
commune    0.104404
EPCI       0.006179
Name: proportion, dtype: float64

- commune → loyer estimé directement au niveau de la commune (le plus précis)
- EPCI → loyer estimé au niveau intercommunalité (précision acceptable)
- maille → loyer estimé sur une maille géographique élargie (plus agrégé mais permet de couvrir 100% du territoire)

Lorsque le nombre d’observations communales est insuffisant, le loyer est estimé à un niveau plus agrégé (EPCI ou maille). La majorité des loyers sont estimés à une maille statistique élargie (89,6 %).
Cela reflète la faible disponibilité d’observations dans les petites communes. Afin de ne pas biaiser l’analyse géographique, je vais conserver toutes les communes et appliquer un indicateur de précision associé.

In [1202]:
#Création d'une colonne "score_precision" basée sur les valeurs de "TYPPRED"
df_cleanLoyer["score_precision"] = df_cleanLoyer["TYPPRED"].map({
    "commune": 3,
    "EPCI": 2,
    "maille": 1})

In [1203]:
df_cleanLoyer.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 419640 entries, 0 to 419639
Data columns (total 16 columns):
 #   Column           Non-Null Count   Dtype  
---  ------           --------------   -----  
 0   id_zone          419640 non-null  object 
 1   INSEE_C          419640 non-null  object 
 2   LIBGEO           419640 non-null  object 
 3   EPCI             419640 non-null  object 
 4   DEP              419640 non-null  object 
 5   REG              419640 non-null  int64  
 6   loypredm2        419640 non-null  float64
 7   lwr.IPm2         419640 non-null  float64
 8   upr.IPm2         419640 non-null  float64
 9   TYPPRED          419640 non-null  object 
 10  nbobs_com        419640 non-null  int64  
 11  nbobs_mail       419640 non-null  int64  
 12  R2_adj           419640 non-null  float64
 13  segment_loyer    419640 non-null  object 
 14  annee            419640 non-null  int64  
 15  score_precision  419640 non-null  int64  
dtypes: float64(4), int64(5), object(7)
mem

In [1204]:
df_cleanLoyer["Code_INSEE"] = df_cleanLoyer["INSEE_C"].astype(str).str.strip().str.zfill(5)

<div style="background-color: #00B6DA; border-radius: 12px;" >
<h3 style="margin: auto; padding: 20px; color:#000000; ">1.3 - Population </h3>
</div>

Afin d’enrichir l’analyse immobilière, une variable démographique a été intégrée au modèle : l’évolution de la population communale entre 2020 et 2023.

In [1205]:
paths_pop = sorted(glob.glob("donnees_communes_20*.csv"))

for p in paths_pop:
    df_test = pd.read_csv(p, sep=";", encoding="utf-8", nrows=3)
    print("\n---", p, "---")
    print(df_test.columns.tolist())
    if "COM" in df_test.columns:
        print("COM sample:", df_test["COM"].head(3).tolist())
    else:
        print("❌ COM absent")


--- donnees_communes_2020.csv ---
['CODREG', 'REG', 'CODDEP', 'CODARR', 'CODCAN', 'CODCOM', 'COM', 'PMUN', 'PCAP', 'PTOT']
COM sample: ["L'Abergement-Clémenciat", "L'Abergement-de-Varey", 'Ambérieu-en-Bugey']

--- donnees_communes_2021.csv ---
['REG', 'Région', 'DEP', 'CODARR', 'CODCAN', 'CODCOM', 'COM', 'Commune', 'PMUN', 'PCAP', 'PTOT']
COM sample: [1001, 1002, 1004]

--- donnees_communes_2022.csv ---
['REG', 'Région', 'DEP', 'CODARR', 'CODCAN', 'CODCOM', 'COM', 'Commune', 'PMUN', 'PCAP', 'PTOT']
COM sample: [1001, 1002, 1004]

--- donnees_communes_2023.csv ---
['REG', 'Région', 'DEP', 'CODARR', 'CODCAN', 'CODCOM', 'COM', 'Commune', 'PMUN', 'PCAP', 'PTOT']
COM sample: [1001, 1002, 1004]


In [1206]:
#Lister les fichiers de population disponibles
paths_pop = sorted(glob.glob("donnees_communes_20*.csv"))
print("Fichiers détectés :", paths_pop)

Fichiers détectés : ['donnees_communes_2020.csv', 'donnees_communes_2021.csv', 'donnees_communes_2022.csv', 'donnees_communes_2023.csv']


In [1207]:
#Vérification des colonnes présentes dans les fichiers de population
for path in paths_pop:
    df_test = pd.read_csv(path, sep=";", encoding="utf-8", nrows=5)
    print(f"\n--- {path} ---")
    print(df_test.columns.tolist())


--- donnees_communes_2020.csv ---
['CODREG', 'REG', 'CODDEP', 'CODARR', 'CODCAN', 'CODCOM', 'COM', 'PMUN', 'PCAP', 'PTOT']

--- donnees_communes_2021.csv ---
['REG', 'Région', 'DEP', 'CODARR', 'CODCAN', 'CODCOM', 'COM', 'Commune', 'PMUN', 'PCAP', 'PTOT']

--- donnees_communes_2022.csv ---
['REG', 'Région', 'DEP', 'CODARR', 'CODCAN', 'CODCOM', 'COM', 'Commune', 'PMUN', 'PCAP', 'PTOT']

--- donnees_communes_2023.csv ---
['REG', 'Région', 'DEP', 'CODARR', 'CODCAN', 'CODCOM', 'COM', 'Commune', 'PMUN', 'PCAP', 'PTOT']


Pour chaque fichier :
- conservation des colonnes pertinentes : CODCOM (code commune) et PTOT (population totale),
- normalisation du code commune sur 5 caractères (zfill),
- ajout d’une colonne annee,
- homogénéisation des noms de colonnes.

Cette étape garantit une structure cohérente pour l’ensemble des années.

In [1208]:
#Fonction de  nettoyage des fichiers population
def load_clean_population(path):
    df = pd.read_csv(path, sep=";", encoding="utf-8")
    df.columns = df.columns.str.strip()

    #Année depuis le nom du fichier
    annee = int(re.search(r"(20\d{2})", path).group(1))

    #Sécuriser types
    df["CODCOM"] = df["CODCOM"].astype(str).str.strip().str.zfill(3)

    #CODDEP existe en 2020, DEP existe en 2021-2023
    if "CODDEP" in df.columns:
        dep = df["CODDEP"].astype(str).str.strip()
    else:
        dep = df["DEP"].astype(str).str.strip()

    #Normalisation département (2A/2B sinon 2 chiffres)
    dep = dep.where(dep.isin(["2A", "2B"]), dep.str.zfill(2))

    #Clé INSEE complète
    df["Code_INSEE"] = dep + df["CODCOM"]

    #Population totale
    df["PTOT"] = pd.to_numeric(df["PTOT"], errors="coerce")

    #Optionnel : nom de commune si disponible
    commune_col = "Commune" if "Commune" in df.columns else None

    keep_cols = ["Code_INSEE", "PTOT"]
    if commune_col:
        keep_cols.append(commune_col)

    df_pop = df[keep_cols].copy()
    df_pop = df_pop.rename(columns={"PTOT": "population"})
    if commune_col:
        df_pop = df_pop.rename(columns={commune_col: "Commune"})

    df_pop["annee"] = annee

    return df_pop

In [1209]:
#Charger et nettoyer tous les fichiers de population, puis concaténer
paths_pop = sorted(glob.glob("donnees_communes_20*.csv"))

dfs_pop = []
for p in paths_pop:
    print("Lecture :", p)
    tmp = load_clean_population(p)
    print(tmp.head(3))
    dfs_pop.append(tmp)

df_population = pd.concat(dfs_pop, ignore_index=True)

print("Shape population :", df_population.shape)
print(df_population["annee"].value_counts().sort_index())
df_population.head()

Lecture : donnees_communes_2020.csv
  Code_INSEE  population  annee
0      01001         821   2020
1      01002         268   2020
2      01004       14662   2020
Lecture : donnees_communes_2021.csv
  Code_INSEE  population                  Commune  annee
0      01001         848  L'Abergement-Clémenciat   2021
1      01002         273    L'Abergement-de-Varey   2021
2      01004       15240        Ambérieu-en-Bugey   2021
Lecture : donnees_communes_2022.csv
  Code_INSEE  population                  Commune  annee
0      01001         875  L'Abergement-Clémenciat   2022
1      01002         279    L'Abergement-de-Varey   2022
2      01004       15930        Ambérieu-en-Bugey   2022
Lecture : donnees_communes_2023.csv
  Code_INSEE  population                  Commune  annee
0      01001         876  L'Abergement-Clémenciat   2023
1      01002         276    L'Abergement-de-Varey   2023
2      01004       16339        Ambérieu-en-Bugey   2023
Shape population : (139810, 4)
annee
2020   

,Code_INSEE,population,annee,Commune
0,01001,821,2020,NaN
1,01002,268,2020,NaN
2,01004,14662,2020,NaN
3,01005,1806,2020,NaN
4,01006,113,2020,NaN


In [1210]:
#Pivotement pour avoir une colonne par année
df_pop_pivot = df_population.pivot_table(
    index="Code_INSEE",
    columns="annee",
    values="population",
    aggfunc="first"
).reset_index()

In [1211]:
#Calcul de la croissance de la population entre 2020 et 2023 et CAGR
df_pop_pivot["croissance_pop_2020_2023"] = (df_pop_pivot[2023] - df_pop_pivot[2020]) / df_pop_pivot[2020]
df_pop_pivot["cagr_pop_2020_2023"] = (df_pop_pivot[2023] / df_pop_pivot[2020]) ** (1/3) - 1

print("NaN croissance :", df_pop_pivot["croissance_pop_2020_2023"].isna().sum())
df_pop_pivot.head()

NaN croissance : 110


annee,Code_INSEE,2020,2021,2022,2023,croissance_pop_2020_2023,cagr_pop_2020_2023
0,01001,821.0,848.0,875.0,876.0,0.066991,0.021850
1,01002,268.0,273.0,279.0,276.0,0.029851,0.009853
2,01004,14662.0,15240.0,15930.0,16339.0,0.114377,0.036758
3,01005,1806.0,1921.0,1941.0,1930.0,0.068660,0.022382
4,01006,113.0,113.0,114.0,115.0,0.017699,0.005865


<div style="background-color: #0078B8; border-radius: 12px;" >
<h2 style="margin: auto; padding: 20px; color:#000000; ">2 - Construction de la table analytique</h2>
</div>

<div style="background-color: #00B6DA; border-radius: 12px;" >
<h3 style="margin: auto; padding: 20px; color:#000000; ">2.1 - Harmonisation des clés de jointure </h3>
</div>

Les sources utilisées (DVF, loyers DHUP, population INSEE) utilisent toutes une clé géographique basée sur le code INSEE commune. Avant toute jointure :
- le code commune est harmonisé au format string sur 5 caractères (zfill(5)),
- ce contrôle garantit l’absence de pertes lors des merges.

In [1212]:
#Vérification des types de données des clés de merge
print(df_cleanVF["Code_INSEE"].dtype)
print(df_cleanLoyer["Code_INSEE"].dtype)
print(df_pop_pivot["Code_INSEE"].dtype)

object
object
object


In [1213]:
#Le code INSEE doit être au format string avec 5 caractères
df_cleanVF["Code_INSEE"] = df_cleanVF["Code_INSEE"].astype(str).str.strip().str.zfill(5)
print(df_cleanVF["Code_INSEE"].dtype)

object


In [1214]:
#Le code INSEE doit être au format string avec 5 caractères
df_cleanLoyer["Code_INSEE"] = df_cleanLoyer["Code_INSEE"].astype(str).str.strip().str.zfill(5)
print(df_cleanLoyer["Code_INSEE"].dtype)

object


In [1215]:
#Le code INSEE des communes doit être au format string avec 5 caractères
df_pop_pivot["Code_INSEE"] = df_pop_pivot["Code_INSEE"].astype(str).str.strip().str.zfill(5)
print(df_pop_pivot["Code_INSEE"].dtype)

object


In [1216]:
# Combien de codes DVF existent dans population ?
ratio = df_cleanVF["Code_INSEE"].isin(df_pop_pivot["Code_INSEE"]).mean()
print("Match DVF -> population :", ratio)

Match DVF -> population : 0.9888600601446742


Validation : 100% des codes communes DVF ont une correspondance dans le référentiel population (match = 1.0).

<div style="background-color: #00B6DA; border-radius: 12px;" >
<h3 style="margin: auto; padding: 20px; color:#000000; ">2.2 - Agrégation DVF </h3>
</div>

Les données DVF étant transactionnelles et volumineuses, elles sont agrégées pour obtenir une table plus adaptée à l’analyse et à Power BI. 

niveau d'agrégation : Commune × Année × Type local (Maison / Appartement)

Indicateurs calculés :
- prix_m2_median : médiane du prix au m² (plus robuste que la moyenne)
- prix_m2_mean : moyenne du prix au m² (à titre informatif)
- nb_ventes : nombre de transactions (mesure de liquidité / fiabilité)
- surface_median : surface médiane
- valeur_median : valeur foncière médiane

Afin de mieux analyser les stratégies d'investissement, trois segments ont été définis : 
- Maison
- Appartement 1-2 pièces
- Appartement 3 pièces et plus

In [1217]:
#Création d'une colonne "segment_bien" pour catégoriser les biens en fonction du type et du nombre de pièces principales
df_cleanVF["segment_bien"] = None

#Maisons
df_cleanVF.loc[
    df_cleanVF["Type local"] == "Maison",
    "segment_bien"] = "Maison"

#Appartements 1–2 pièces
df_cleanVF.loc[
    (df_cleanVF["Type local"] == "Appartement") &
    (df_cleanVF["Nombre pieces principales"] <= 2),
    "segment_bien"] = "App_1_2"

#Appartements 3+ pièces
df_cleanVF.loc[
    (df_cleanVF["Type local"] == "Appartement") &
    (df_cleanVF["Nombre pieces principales"] >= 3),
    "segment_bien"] = "App_3_plus"

df_cleanVF["segment_bien"].value_counts()

segment_bien
Maison        2630633
App_3_plus     993747
App_1_2        975196
Name: count, dtype: int64

In [1218]:
#Agrégation des données DVF par commune, année et segment de bien
df_aggVF = (
    df_cleanVF
    .groupby([
        "Code_INSEE",
        "annee_source",
        "segment_bien"])
    .agg(
        prix_m2_median=("prix_m2", "median"),
        prix_m2_mean=("prix_m2", "mean"),
        nb_ventes=("prix_m2", "count"),
        surface_median=("Surface reelle bati", "median"),
        valeur_median=("Valeur fonciere", "median"))
    .reset_index()
    .rename(columns={"annee_source": "annee"}))

print("DVF agrégé :", df_aggVF.shape)
df_aggVF.head()

DVF agrégé : (191724, 8)


,Code_INSEE,annee,segment_bien,prix_m2_median,prix_m2_mean,nb_ventes,surface_median,valeur_median
0,01001,2021,Maison,2539.932111,2570.672952,14,121.5,313120.0
1,01001,2022,Maison,2812.500000,2800.050620,13,107.0,282000.0
2,01001,2023,Maison,2345.132743,3220.607801,15,90.0,270000.0
3,01001,2024,Maison,3212.087912,3248.685531,5,91.0,292300.0
4,01002,2021,App_3_plus,4090.909091,4226.539589,3,110.0,450000.0


<div style="background-color: #00B6DA; border-radius: 12px;" >
<h3 style="margin: auto; padding: 20px; color:#000000; ">2.3 - Agrégation Loyer</h3>
</div>

Les loyers sont fournis sous plusieurs segments :
- APP : appartements (tous)
- APP12 : appartements 1–2 pièces
- APP3 : appartements 3 pièces
- MAI : maisons

Afin de pouvoir relier loyers et DVF, les segments sont harmonisés en 3 catégories :
- APP12 → App_1_2
- APP3 → App_3_plus
- MAIS → Maison

Niveau d’agrégation : Commune × Année × Segment du bien

In [1219]:
#Filtrage des données de loyer pour ne garder que les segments d'appartements 1-2 pièces, 3+ pièces et maisons
df_loyer_seg = df_cleanLoyer[df_cleanLoyer["segment_loyer"].isin(["APP12", "APP3", "MAI"])].copy()

In [1220]:
#Création d'une colonne "segment_bien" dans les données de loyer pour correspondre aux segments DVF
df_loyer_seg["segment_bien"] = df_loyer_seg["segment_loyer"].map({
    "APP12": "App_1_2",
    "APP3": "App_3_plus",
    "MAI": "Maison"})

In [1221]:
#Agrégation des données de loyer par commune, année et segment de bien
df_aggLoyer = (
    df_loyer_seg
    .groupby(["Code_INSEE", "annee", "segment_bien"])
    .agg(
        loyer_m2=("loypredm2", "median"),
        nbobs_com=("nbobs_com", "sum"))
    .reset_index())

print("Loyers agrégés :", df_aggLoyer.shape)
df_aggLoyer.head()

Loyers agrégés : (314730, 5)


,Code_INSEE,annee,segment_bien,loyer_m2,nbobs_com
0,01001,2022,App_1_2,13.387844,8
1,01001,2022,App_3_plus,8.956710,0
2,01001,2022,Maison,8.638046,24
3,01001,2023,App_1_2,11.777844,6
4,01001,2023,App_3_plus,9.488145,0


<div style="background-color: #00B6DA; border-radius: 12px;" >
<h3 style="margin: auto; padding: 20px; color:#000000; ">2.4 - Merge DVF et Loyer</h3>
</div>

In [1222]:
#Affichage des colonnes des DataFrames agrégés pour vérifier les clés de merge
print(df_aggVF.columns)
print(df_aggLoyer.columns)

Index(['Code_INSEE', 'annee', 'segment_bien', 'prix_m2_median', 'prix_m2_mean',
       'nb_ventes', 'surface_median', 'valeur_median'],
      dtype='object')
Index(['Code_INSEE', 'annee', 'segment_bien', 'loyer_m2', 'nbobs_com'], dtype='object')


In [1223]:
#Affichage de l'intersection des codes INSEE entre les données DVF agrégées et les données de loyer agrégées
print("Intersection codes INSEE :",
      len(set(df_aggVF["Code_INSEE"]) & set(df_aggLoyer["Code_INSEE"])))

Intersection codes INSEE : 33069


In [1224]:
#Fusion des données DVF agrégées et des données de loyer agrégées sur les clés "Code_INSEE", "annee" et "segment_bien"
df_model = df_aggVF.merge(
    df_aggLoyer,
    on=["Code_INSEE", "annee", "segment_bien"],
    how="inner")

print("Shape modèle :", df_model.shape)
print(df_model["annee"].value_counts().sort_index())

Shape modèle : (141317, 10)
annee
2022    49032
2023    46705
2024    45580
Name: count, dtype: int64


In [1225]:
#Fusion des données de population pivotées avec le modèle sur la clé "Code_INSEE" pour ajouter les colonnes de croissance de la population
df_model = df_model.merge(
    df_pop_pivot[["Code_INSEE", "croissance_pop_2020_2023", "cagr_pop_2020_2023"]],
    on="Code_INSEE",
    how="left",
    validate="m:1")

print("NaN croissance pop dans df_model :", df_model["croissance_pop_2020_2023"].isna().sum())

NaN croissance pop dans df_model : 263


Quelques codes INSEE (≈0,2%) ne sont pas retrouvés sur l’ensemble de la période 2020–2023, probablement en raison de fusions ou modifications administratives. Les lignes sont conservées, avec un indicateur pop_missing afin d’assurer la transparence des données.

In [1226]:
df_model["pop_missing"] = df_model["croissance_pop_2020_2023"].isna().astype(int)
df_model["pop_missing"].value_counts()

pop_missing
0    141054
1       263
Name: count, dtype: int64

Les données INSEE 2020–2023 ont permis de calculer :
- croissance_pop_2020_2023
- cagr_pop_2020_2023

Ces indicateurs permettent d’évaluer :
- l’attractivité structurelle d’une commune,
- la cohérence entre hausse des prix et croissance démographique,
- le risque locatif potentiel.

<div style="background-color: #00B6DA; border-radius: 12px;" >
<h3 style="margin: auto; padding: 20px; color:#000000; ">2.5 - Construction du modèle final</h3>
</div>

In [1227]:
#Ajout d'une colonne de rendement brut en pourcentage : (loyer_m2 * 12 / prix_m2_median) * 100
df_model["rendement_brut_pct"] = (df_model["loyer_m2"] * 12 / df_model["prix_m2_median"]) * 100

Le rendement brut est calculé comme suit : rendement brut (%) = ((loyer m² * 12) / prix m²) * 100 

C'est une estimation brute qui ne prend pas en compte : 
- la fiscalité
- les charges
- les travaux
- le financement

Il permet néanmoins une comparaison relative entre communes et segments.

Les données INSEE 2020–2023 ont permis de calculer :
- croissance_pop_2020_2023
- cagr_pop_2020_2023

Ces indicateurs permettent d’évaluer :
- l’attractivité structurelle d’une commune,
- la cohérence entre hausse des prix et croissance démographique,
- le risque locatif potentiel.

In [1228]:
df_model.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 141317 entries, 0 to 141316
Data columns (total 14 columns):
 #   Column                    Non-Null Count   Dtype  
---  ------                    --------------   -----  
 0   Code_INSEE                141317 non-null  object 
 1   annee                     141317 non-null  int64  
 2   segment_bien              141317 non-null  object 
 3   prix_m2_median            141317 non-null  float64
 4   prix_m2_mean              141317 non-null  float64
 5   nb_ventes                 141317 non-null  int64  
 6   surface_median            141317 non-null  float64
 7   valeur_median             141317 non-null  float64
 8   loyer_m2                  141317 non-null  float64
 9   nbobs_com                 141317 non-null  int64  
 10  croissance_pop_2020_2023  141054 non-null  float64
 11  cagr_pop_2020_2023        141054 non-null  float64
 12  pop_missing               141317 non-null  int64  
 13  rendement_brut_pct        141317 non-null  f

In [1229]:
#Filtrage pour ne garder que les segments avec au moins 10 ventes (nb_ventes >= 10) pour assurer une certaine fiabilité des prix médians
df_model = df_model[df_model["nb_ventes"] >= 10].copy()

In [1230]:
#Structure finale du DataFrame prêt pour l'analyse et la modélisation
print("Shape df_model :", df_model.shape)
print(df_model.dtypes)
df_model.head()

Shape df_model : (57015, 14)
Code_INSEE                   object
annee                         int64
segment_bien                 object
prix_m2_median              float64
prix_m2_mean                float64
nb_ventes                     int64
surface_median              float64
valeur_median               float64
loyer_m2                    float64
nbobs_com                     int64
croissance_pop_2020_2023    float64
cagr_pop_2020_2023          float64
pop_missing                   int64
rendement_brut_pct          float64
dtype: object


,Code_INSEE,annee,segment_bien,prix_m2_median,prix_m2_mean,nb_ventes,surface_median,valeur_median,loyer_m2,nbobs_com,croissance_pop_2020_2023,cagr_pop_2020_2023,pop_missing,rendement_brut_pct
0,01001,2022,Maison,2812.500000,2800.050620,13,107.0,282000.0,8.638046,24,0.066991,0.021850,0,3.685566
1,01001,2023,Maison,2345.132743,3220.607801,15,90.0,270000.0,9.641002,29,0.066991,0.021850,0,4.933282
7,01004,2022,App_1_2,2822.332641,4656.777788,42,44.5,137500.0,13.076444,807,0.114377,0.036758,0,5.559845
8,01004,2022,App_3_plus,2279.411765,3140.841106,87,72.0,164000.0,9.377498,1014,0.114377,0.036758,0,4.936799
9,01004,2022,Maison,2616.666667,2716.105612,139,93.0,235250.0,8.825813,321,0.114377,0.036758,0,4.047507


In [1231]:
#Vérification des doublons sur la clé de fusion (Code_INSEE, annee, segment_bien)
key_cols = ["Code_INSEE", "annee", "segment_bien"]

nb_dup = df_model.duplicated(subset=key_cols).sum()
print("Doublons sur clé (Code_INSEE/annee/segment) :", nb_dup)

Doublons sur clé (Code_INSEE/annee/segment) : 0


In [1232]:
#Vérification des valeurs manquantes dans les colonnes critiques pour l'analyse
cols_crit = ["prix_m2_median", "loyer_m2", "rendement_brut_pct", "croissance_pop_2020_2023"]

print(df_model[cols_crit].isna().sum())

prix_m2_median               0
loyer_m2                     0
rendement_brut_pct           0
croissance_pop_2020_2023    74
dtype: int64


In [1233]:
#Statistiques descriptives du rendement brut en pourcentage
print(df_model["rendement_brut_pct"].describe(percentiles=[0.01,0.05,0.5,0.95,0.99]))

count    57015.000000
mean         5.336970
std          1.716142
min          0.417269
1%           1.803130
5%           3.013510
50%          5.132305
95%          8.303178
99%         10.597594
max         38.393914
Name: rendement_brut_pct, dtype: float64


In [1234]:
#Vérification de la répartition des observations par année et segment de bien
print(df_model["annee"].value_counts().sort_index())
print(df_model["segment_bien"].value_counts())
print(df_model.groupby(["annee", "segment_bien"]).size())

annee
2022    21403
2023    18396
2024    17216
Name: count, dtype: int64
segment_bien
Maison        42546
App_3_plus     7344
App_1_2        7125
Name: count, dtype: int64
annee  segment_bien
2022   App_1_2          2627
       App_3_plus       2770
       Maison          16006
2023   App_1_2          2345
       App_3_plus       2388
       Maison          13663
2024   App_1_2          2153
       App_3_plus       2186
       Maison          12877
dtype: int64


In [1235]:
df_model.to_csv("df_model_final.csv", index=False)